In [ ]:
from langchain_community.document_loaders import PyPDFLoader

Number of Document objects: 256

First Document object:
page_content='' metadata={'producer': 'calibre 3.48.0 [https://calibre-ebook.com]', 'creator': 'calibre 3.48.0 [https://calibre-ebook.com]', 'creationdate': '2020-04-30T18:46:22+00:00', 'author': 'James Clear', 'title': 'Atomic habits \\( PDFDrive.com \\).pdf', 'source': 'sample.pdf', 'total_pages': 256, 'page': 1, 'page_label': '2'}


In [ ]:
loader = PyPDFLoader("sample.pdf")   # use any real pdf you've already tested with
documents = loader.load()

print(f"Number of Document objects: {len(documents)}")
print()
print("First Document object:")
print(documents[1])

In [6]:
import os
from docx2pdf import convert

In [7]:
def _process_pdf_or_docx(file_path: str) -> list:
    chunks = []
    pdf_to_read = file_path
    temp_pdf = False

    if file_path.endswith(".docx"):
        pdf_to_read = file_path.replace(".docx", "_temp.pdf")
        convert(file_path, pdf_to_read)
        temp_pdf = True

    loader = PyPDFLoader(pdf_to_read)
    documents = loader.load()   # one Document per page

    for doc in documents:
        text = doc.page_content.strip()
        page_num = doc.metadata["page"] + 1   #langChains page count starts at 0 so +1 to match real page numbers
        if text:
            for start in range(0, len(text), 5000):
                part_num = start // 5000 + 1
                chunks.append({
                    "text": text[start:start + 5000],
                    "source": f"Page {page_num} (part {part_num})"
                })
    # --------------------------------------------

    if temp_pdf and os.path.exists(pdf_to_read):
        os.remove(pdf_to_read)

    return chunks

In [1]:
from langchain_community.document_loaders import UnstructuredPowerPointLoader


C:\Users\BM\AppData\Local\Temp\ipykernel_11084\810733367.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredPowerPointLoader


In [ ]:
loader = UnstructuredPowerPointLoader("changeControl.pptx")
documents = loader.load()

print(f"Number of Document objects: {len(documents)}")
for doc in documents:
    print(doc.metadata)

In [ ]:
from langchain_community.document_loaders import UnstructuredPowerPointLoader

loader = UnstructuredPowerPointLoader("changeControl.pptx", strategy="fast")
documents = loader.load()

print(f"Number of Document objects: {len(documents)}")

C:\Users\BM\AppData\Local\Temp\ipykernel_3564\3656908264.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredPowerPointLoader


In [2]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\BM\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\BM\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.


True

In [3]:
from langchain_community.document_loaders import UnstructuredExcelLoader

loader = UnstructuredExcelLoader("test_multi_sheet.xlsx", mode="elements")
documents = loader.load()

print(f"Number of Documents: {len(documents)}")
for doc in documents:
    print(doc.metadata)
    print(doc.page_content[:200])
    print("---")

Number of Documents: 3
{'source': 'test_multi_sheet.xlsx', 'filename': 'test_multi_sheet.xlsx', 'last_modified': '2026-08-15T01:55:39', 'page_name': 'Employee Directory', 'page_number': 1, 'text_as_html': '<table><tr><td>Emp ID</td><td>Full Name</td><td>Department</td><td>Role</td><td>Location</td><td>Status</td></tr><tr><td>EMP-101</td><td>Amina Khan</td><td>Engineering</td><td>Backend Developer</td><td>Lahore</td><td>Active</td></tr><tr><td>EMP-102</td><td>Bilal Ahmed</td><td>Data Science</td><td>AI Engineer</td><td>Karachi</td><td>Active</td></tr><tr><td>EMP-103</td><td>Chaudhry Riaz</td><td>Human Resources</td><td>HR Manager</td><td>Islamabad</td><td>Active</td></tr><tr><td>EMP-104</td><td>Dania Zafar</td><td>Marketing</td><td>SEO Specialist</td><td>Lahore</td><td>On Leave</td></tr></table>', 'languages': ['eng'], 'filetype': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet', 'category': 'Table', 'element_id': '0bf930430e68e37950049f3078aefda1'}
Emp ID Full Name D

In [9]:
def _split_capped(text: str, label_prefix: str, chunks: list):
    for start in range(0, len(text), 5000):
        part_num = start // 5000 + 1
        piece = text[start:start + 5000]
        chunks.append({"text": piece, "source": f"{label_prefix} (part {part_num})"})


def _process_xlsx(file_path: str) -> list:
    """
    NOTE: unverified as of migration. Confirm this actually returns
    content from ALL sheets (not just the first) before trusting it --
    that was the core bug in the original manual pandas version.
    """
    chunks = []
    loader = UnstructuredExcelLoader(file_path, mode="elements")
    documents = loader.load()

    for doc in documents:
        text = doc.page_content.strip()
        sheet_name = doc.metadata.get("page_name", "unknown_sheet")  # verify this key via printed metadata
        if text:
            _split_capped(text, f"Sheet '{sheet_name}'", chunks)

    return chunks

In [15]:
# chunks =  _process_pdf_or_docx("sample.pdf")
# chunks = process_file("changeControl.pptx")
# chunks = process_file("CarromRules.docx")
# chunks = process_file("DSA_PracticeQs.xlsx")
# chunks = _process_xlsx("test_multi_sheet.xlsx")
chunks = _process_xlsx("DSA_PracticeQs.xlsx")
print(f"Total chunks extracted: {len(chunks)}")
print("\nFirst Chunk: ")
print(f"Source: {chunks[2]['source']}")
print(f"Content:\n{chunks[2]['text'][:600]}...")

Total chunks extracted: 10

First Chunk: 
Source: Sheet 'Sheet1' (part 1)
Content:
True 6.0 Kadane's Algorithm Problem Link Medium Solution Link Microsoft Facebook True 7.0 Pow xn Problem Link Medium Solution Link True 8.0 Container with most water Problem Link Medium Solution Link Flipkart Dunzo True 9.0 Sort array of 0s, 1s & 2s Problem Link Medium Solution Link Microsoft Amazon MakeMyTrip Sorting True 10.0 3Sum Problem Link Medium Solution Link Adobe Amazon Microsoft Morgan Stanley Samsung Snapdeal Times Internet Hashing False 11.0 4Sum Problem Link Medium Solution Link Hashing False 12.0 Search a 2D matrix Problem Link Medium Solution Link 2D Array False 13.0 Next permut...


In [ ]:
import sys
sys.path.append("..")

from src.vectorstore.milvus_store import _client, COLLECTION_NAME

results = _client.query(collection_name=COLLECTION_NAME, filter="", output_fields=["filename"], limit=10000)
print(len(results), "total chunks")
filenames = {r["filename"] for r in results}
print(filenames)

In [1]:
print("helllo")

helllo


In [2]:
import os, sys
os.chdir("..")
sys.path.append(os.getcwd())

print("Now running from:", os.getcwd())
print("Does rag_milvus.db exist here?", os.path.exists("rag_milvus.db"))

Now running from: d:\RAG
Does rag_milvus.db exist here? True


In [5]:
from src.vectorstore.milvus_store import _client, COLLECTION_NAME

_client.load_collection(collection_name=COLLECTION_NAME)

results = _client.query(collection_name=COLLECTION_NAME, filter="", output_fields=["filename"], limit=10000)
print(len(results), "total chunks")
filenames = {r["filename"] for r in results}
print(filenames)

8 total chunks
{'CarromRules.docx', 'DSA-GUIDE.pdf'}
